# MeChess 11/11: model games from the human starting positions (GPU notebook)

An adapter around `chessme humanlike-selfplay --starts-from`. It plays the **same starting positions** as notebook 10's human games (position, clocks and ratings of the real game), with a Maia-3 model against itself, so that human and generated games can be compared one to one (the classifier two-sample test, and the stimuli of the forced-choice study). `PLAYER` chooses who plays: the frozen model, the likelihood adapter or the behaviour-matched adapter of notebook 9. Moves are sampled from the model's probabilities; thinking time is drawn from what humans spent in the same situation. Run it once per player (two GPU sessions can run side by side).

**Before you run:** add as inputs the outputs of notebook 6 (test sets), notebook 7 (scores), notebook 9 (the saved adapters) and notebook 10 (the human games); *Accelerator -> GPU*, *Internet -> On*; **Save Version -> Save & Run All (Commit)**. Maia-3 (code and weights) is an external download (AGPL-3.0) and is not redistributed.

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here (not needed when the repository is attached as a dataset)
MODEL = "23m"                                           # which Maia-3 size to adapt: 5m, 23m or 79m (notebooks 7 and 8 must have been run for it)
GAMES = 6000                                            # games (the first GAMES starting positions of the human games)
PLAYER = "behaviour"                                    # "frozen", "likelihood" or "behaviour"
TEMPERATURE = 1.0                                       # 1: the model as it is; below 1 sharpened, above 1 flattened (dose-response control of the classifier test)
TAG = PLAYER if TEMPERATURE == 1.0 else f"{PLAYER}_T{TEMPERATURE:g}"
MAIA3_COMMIT = ""                                       # empty: the current default branch; a commit hash pins the baseline exactly
OUT = "/kaggle/working/humanlike" if os.path.exists("/kaggle") else "humanlike_local"
INPUT_DIR = ""                                          # the output folders of notebooks 6, 7, 9 and 10; empty: search /kaggle/input

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    attached = sorted(glob.glob("/kaggle/input/**/chessme/__init__.py", recursive=True), key=lambda f: "/datasets/" not in f)   # prefer the attached repository dataset
    if attached:
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), "MeChess")
    else:                                                           # Kaggle: fetch the repository (Internet must be on)
        if "<you>" in REPO_URL:
            raise SystemExit("Attach the repository as a dataset, or set REPO_URL.")
        sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "chess==1.11.2", "numpy", "pyyaml", "requests", "zstandard")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

In [ ]:
root = INPUT_DIR or "/kaggle/input"
def find(name):
    return next(iter(sorted(glob.glob(root + "/**/" + name, recursive=True))), None)
TB, HUMAN = find("testset_TB.jsonl.gz"), find("games_human.jsonl")
ADAPTER = {"frozen": None, "likelihood": find("adapter_all_features_temperature_tail_slot_biases.pt"), "behaviour": find("behaviour_matched_adapter_lambda_0.5.pt")}[PLAYER]
print(TB, HUMAN, ADAPTER, sep="\n")
if not (TB and HUMAN) or (PLAYER != "frozen" and not ADAPTER):
    raise SystemExit("Add the outputs of notebooks 6, 9 (the saved adapters) and 10 (the human games) as inputs.")
os.makedirs(OUT, exist_ok=True)

## Step 1. Maia-3 (external, not part of this repository)

In [ ]:
sh("git", "clone", "--depth", "1", "https://github.com/CSSLab/maia3.git", "/tmp/maia3")
if MAIA3_COMMIT:
    sh("git", "-C", "/tmp/maia3", "fetch", "--depth", "1", "origin", MAIA3_COMMIT)
    sh("git", "-C", "/tmp/maia3", "checkout", MAIA3_COMMIT)
sh("git", "-C", "/tmp/maia3", "log", "-1", "--format=maia3 commit %h %ad")
os.makedirs("/tmp/weights", exist_ok=True)
sh("curl", "-fsSL", "-o", f"/tmp/weights/maia3-{MODEL}.pt", f"https://huggingface.co/UofTCSSLab/Maia3-{MODEL.upper()}/resolve/main/maia3-{MODEL}.pt")
sh("sha256sum", f"/tmp/weights/maia3-{MODEL}.pt")

## Step 2. The games

In [ ]:
SCORER = f"maia3={MODEL}+history"
ADAPTED = ["--white-adapter", ADAPTER, "--black-adapter", ADAPTER] if ADAPTER else []
sh(*CLI, "humanlike-selfplay", "--testset", TB, "--maia3-repo", "/tmp/maia3", "--weights-dir", "/tmp/weights", "--device", "cuda", "--games", str(GAMES), "--split", "test",
   "--seed", "1", "--starts-from", HUMAN, "--white", SCORER, "--black", SCORER, *ADAPTED, "--temperature", str(TEMPERATURE), "--out", f"{OUT}/games_{TAG}.jsonl")

## Step 3. Statistics, against the real continuations of the same starts (the human games)

In [ ]:
sh(*CLI, "humanlike-gamestats", "--games", f"{TAG}={OUT}/games_{TAG}.jsonl", "--out", f"{OUT}/gamestats_{TAG}.md")